 # NLI with Traditional Machine Learning

 Binary classification: given a **premise** and **hypothesis**, predict
 whether the hypothesis is entailed (1) or not (0).

 Pipeline:
 1. Baseline: TF-IDF on concatenated text + Logistic Regression
 2. Feature-engineered: cross-sentence features (TF-IDF + linguistic + alignment)
 3. Model comparison (LR, LinearSVC, XGBoost, Stacking Ensemble)
 4. Hyperparameter tuning
 5. Ablation study
 6. Error analysis with statistical significance testing

Notes (23/03/2026):
Stacking currently uses OOF meta-features.
Final deployable stacking bundle and inference function will be added after test data release.

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import warnings
from pathlib import Path
from collections import Counter

import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

### Data Loading

In [ ]:
DATA_DIR = Path("training_data/training_data/NLI")
TRIAL_PATH = Path("trial_data (1)/trial_data/NLI_trial.csv")

train_df = pd.read_csv(DATA_DIR / "train.csv")
dev_df   = pd.read_csv(DATA_DIR / "dev.csv")

print(f"Train: {len(train_df):,}  |  Dev: {len(dev_df):,}")
print(train_df.head(3))
print("\nLabel distribution (train):")
print(train_df['label'].value_counts(normalize=True).round(3))


## 1. Baseline — TF-IDF + Logistic Regression

In [ ]:
X_train_raw = (train_df['premise'].fillna('') + ' [SEP] ' + train_df['hypothesis'].fillna(''))
X_dev_raw   = (dev_df['premise'].fillna('')   + ' [SEP] ' + dev_df['hypothesis'].fillna(''))
y_train = train_df['label']
y_dev   = dev_df['label']

tfidf_concat = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, sublinear_tf=True)
X_train_tfidf = tfidf_concat.fit_transform(X_train_raw)
X_dev_tfidf   = tfidf_concat.transform(X_dev_raw)

lr_baseline = LogisticRegression(max_iter=1000, C=1.0)
lr_baseline.fit(X_train_tfidf, y_train)
y_pred_base = lr_baseline.predict(X_dev_tfidf)

print("=== Baseline: TF-IDF (concat) + Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_dev, y_pred_base):.4f}")
print(classification_report(y_dev, y_pred_base))

## 2. Enhanced Feature Engineering

 We build **four groups** of handcrafted features — all firmly within
 Category A (no pre-trained embeddings or deep learning):

 | Group | Features | Rationale |
 |---|---|---|
 | **Lexical overlap** | Word overlap recall/precision, Jaccard, BLEU-1/2 | Surface-level alignment |
 | **Linguistic** | NER overlap, WordNet synonyms/antonyms/hypernyms | Semantic relationships via knowledge bases |
 | **Sentiment / polarity** | VADER compound score diff, polarity mismatch | Contradiction often flips sentiment |
 | **String / structural** | Length ratio, edit distance, longest common subsequence, negation flags | Structural cues |



In [ ]:
import nltk
for resource in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger',
                 'averaged_perceptron_tagger_eng', 'punkt', 'punkt_tab',
                 'vader_lexicon', 'stopwords']:
    nltk.download(resource, quiet=True)

from nltk.corpus import wordnet as wn, stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

STOP_WORDS = set(stopwords.words('english'))
sia = SentimentIntensityAnalyzer()

### Tokenisation Helpers

In [ ]:
import re

NEGATION_WORDS = {
    'no', 'not', 'never', 'neither', 'nor', "n't", 'nobody',
    'nothing', 'nowhere', 'hardly', 'scarcely', 'barely', 'without',
    'isn', 'aren', 'wasn', 'weren', 'hasn', 'haven', 'hadn',
    'doesn', 'didn', 'won', 'wouldn', 'shouldn', 'couldn', 'mustn'
}

def tokenize_simple(text):
    """Lowercase, strip punctuation, split on whitespace."""
    return re.sub(r"[^a-z0-9\s]", " ", str(text).lower()).split()

### WordNet-based features
 These capture semantic relationships (synonymy, antonymy, hypernymy)
 using a purely linguistic knowledge base — no embeddings involved.
 Grounded in MacCartney & Manning (2007) natural logic for NLI.

In [ ]:
def get_synsets(word):
    """Get all WordNet synsets for a word."""
    return wn.synsets(word)

def get_synonyms(word):
    """Get all synonyms from WordNet."""
    synonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().lower().replace('_', ' '))
    return synonyms

def get_antonyms(word):
    """Get all antonyms from WordNet."""
    antonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            for ant in lemma.antonyms():
                antonyms.add(ant.name().lower().replace('_', ' '))
    return antonyms

def get_hypernyms(word):
    """Get immediate hypernyms (is-a parents)."""
    hypernyms = set()
    for syn in wn.synsets(word):
        for hyper in syn.hypernyms():
            for lemma in hyper.lemmas():
                hypernyms.add(lemma.name().lower().replace('_', ' '))
    return hypernyms

def wordnet_features(p_tokens, h_tokens):
    """
    Compute WordNet-based features between premise and hypothesis.
    Returns: [synonym_overlap, antonym_overlap, hypernym_overlap,
              avg_wup_similarity, max_wup_similarity]
    """
    p_content = [w for w in p_tokens if w not in STOP_WORDS and w.isalpha()]
    h_content = [w for w in h_tokens if w not in STOP_WORDS and w.isalpha()]

    if not h_content or not p_content:
        return [0.0, 0.0, 0.0, 0.0, 0.0]

    # Build premise synonym, antonym, and hypernym sets
    p_synonyms = set()
    p_antonyms = set()
    p_hypernyms = set()
    for w in p_content:
        p_synonyms.update(get_synonyms(w))
        p_antonyms.update(get_antonyms(w))
        p_hypernyms.update(get_hypernyms(w))

    # Count hypothesis words that match premise synonyms / antonyms / hypernyms
    synonym_hits = sum(1 for w in h_content if w in p_synonyms)
    antonym_hits = sum(1 for w in h_content if w in p_antonyms)
    hypernym_hits = sum(1 for w in h_content if w in p_hypernyms)

    synonym_overlap = synonym_hits / len(h_content)
    antonym_overlap = antonym_hits / len(h_content)
    hypernym_overlap = hypernym_hits / len(h_content)

    # Wu-Palmer similarity: for each hypothesis word, find best-matching
    # premise word by WuP score (captures taxonomic closeness)
    wup_scores = []
    for hw in h_content[:15]:  # limit for speed
        h_syns = wn.synsets(hw)
        if not h_syns:
            continue
        best_wup = 0.0
        for pw in p_content[:15]:
            p_syns = wn.synsets(pw)
            for hs in h_syns[:2]:
                for ps in p_syns[:2]:
                    score = hs.wup_similarity(ps)
                    if score and score > best_wup:
                        best_wup = score
        wup_scores.append(best_wup)

    avg_wup = np.mean(wup_scores) if wup_scores else 0.0
    max_wup = max(wup_scores) if wup_scores else 0.0

    return [synonym_overlap, antonym_overlap, hypernym_overlap, avg_wup, max_wup]



### Sentiment / polarity features
Contradiction often involves a sentiment flip. VADER is rule-based
(no deep learning)

In [ ]:
def sentiment_features(p_text, h_text):
    """
    Compare VADER sentiment between premise and hypothesis.
    Returns: [compound_diff, abs_compound_diff, polarity_mismatch]
    """
    p_scores = sia.polarity_scores(str(p_text))
    h_scores = sia.polarity_scores(str(h_text))

    compound_diff = p_scores['compound'] - h_scores['compound']
    abs_diff = abs(compound_diff)

    # Polarity mismatch: one positive, the other negative
    p_polarity = 1 if p_scores['compound'] > 0.05 else (-1 if p_scores['compound'] < -0.05 else 0)
    h_polarity = 1 if h_scores['compound'] > 0.05 else (-1 if h_scores['compound'] < -0.05 else 0)
    polarity_mismatch = int(p_polarity * h_polarity == -1)

    return [compound_diff, abs_diff, polarity_mismatch]


### String alignment features

In [ ]:
def bleu_n(p_tokens, h_tokens, n):
    """Compute simple BLEU-n precision (no brevity penalty)."""
    if len(h_tokens) < n:
        return 0.0
    p_ngrams = Counter(zip(*[p_tokens[i:] for i in range(n)]))
    h_ngrams = Counter(zip(*[h_tokens[i:] for i in range(n)]))
    matches = sum((h_ngrams & p_ngrams).values())
    total = sum(h_ngrams.values())
    return matches / total if total else 0.0

def lcs_length(a, b):
    """Length of longest common subsequence (token-level)."""
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    # Limit to avoid excessive computation
    a, b = a[:50], b[:50]
    m, n = len(a), len(b)
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                curr[j] = prev[j-1] + 1
            else:
                curr[j] = max(curr[j-1], prev[j])
        prev = curr
    return prev[n]

def edit_distance_norm(a, b):
    """Normalised token-level Levenshtein distance."""
    a, b = a[:50], b[:50]
    m, n = len(a), len(b)
    if m == 0 and n == 0:
        return 0.0
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            if a[i-1] == b[j-1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n] / max(m, n)

def string_alignment_features(p_tokens, h_tokens):
    """
    Returns: [bleu1, bleu2, lcs_ratio, edit_dist_norm]
    """
    b1 = bleu_n(p_tokens, h_tokens, 1)
    b2 = bleu_n(p_tokens, h_tokens, 2)
    lcs_r = lcs_length(p_tokens, h_tokens) / len(h_tokens) if h_tokens else 0.0
    ed = edit_distance_norm(p_tokens, h_tokens)
    return [b1, b2, lcs_r, ed]

### Combined feature extraction

In [ ]:
def enhanced_handcrafted_features(df, verbose=True):
    """
    Extract ALL handcrafted features for a dataframe.
    Returns numpy array of shape (n_samples, n_features).
    """
    all_rows = []
    n = len(df)

    for i, (_, row) in enumerate(df.iterrows()):
        if verbose and (i + 1) % 2000 == 0:
            print(f"  Processing {i+1}/{n}...")

        p_text = str(row['premise'])
        h_text = str(row['hypothesis'])
        p_tokens = tokenize_simple(p_text)
        h_tokens = tokenize_simple(h_text)
        p_set = set(p_tokens)
        h_set = set(h_tokens)

        # ── GROUP 1: Lexical overlap (original + BLEU) ──
        overlap = len(p_set & h_set)
        recall_h = overlap / len(h_set) if h_set else 0.0
        precision_p = overlap / len(p_set) if p_set else 0.0
        union = len(p_set | h_set)
        jaccard = overlap / union if union else 0.0

        # Content word overlap (excluding stopwords)
        p_content = p_set - STOP_WORDS
        h_content = h_set - STOP_WORDS
        content_overlap = len(p_content & h_content)
        content_recall = content_overlap / len(h_content) if h_content else 0.0

        # New word ratio (hypothesis words NOT in premise)
        new_words = len(h_set - p_set)
        new_word_ratio = new_words / len(h_set) if h_set else 0.0

        # ── GROUP 2: String alignment ──
        alignment = string_alignment_features(p_tokens, h_tokens)

        # ── GROUP 3: Structural / length ──
        len_p = len(p_tokens)
        len_h = len(h_tokens)
        len_ratio = len_h / len_p if len_p else 0.0
        len_diff = len_p - len_h
        char_len_p = len(p_text)
        char_len_h = len(h_text)
        char_ratio = char_len_h / char_len_p if char_len_p else 0.0

        # ── GROUP 4: Negation ──
        neg_p = int(bool(p_set & NEGATION_WORDS))
        neg_h = int(bool(h_set & NEGATION_WORDS))
        neg_mismatch = int(neg_p != neg_h)

        # ── GROUP 5: WordNet features ──
        wn_feats = wordnet_features(p_tokens, h_tokens)

        # ── GROUP 6: Sentiment ──
        sent_feats = sentiment_features(p_text, h_text)

        # ── Combine all ──
        feature_vector = [
            # Lexical overlap (7 features)
            recall_h, precision_p, jaccard, content_recall, new_word_ratio,
            overlap, content_overlap,
            # String alignment (4 features)
            *alignment,
            # Structural (7 features)
            len_p, len_h, len_ratio, len_diff,
            char_len_p, char_len_h, char_ratio,
            # Negation (3 features)
            neg_p, neg_h, neg_mismatch,
            # WordNet (5 features)
            *wn_feats,
            # Sentiment (3 features)
            *sent_feats,
        ]
        all_rows.append(feature_vector)

    return np.array(all_rows, dtype=np.float32)

# Feature names for ablation / analysis
FEATURE_NAMES = [
    # Lexical overlap
    'word_recall_h', 'word_precision_p', 'jaccard', 'content_recall',
    'new_word_ratio', 'overlap_count', 'content_overlap_count',
    # String alignment
    'bleu_1', 'bleu_2', 'lcs_ratio', 'edit_dist_norm',
    # Structural
    'len_p', 'len_h', 'len_ratio', 'len_diff',
    'char_len_p', 'char_len_h', 'char_ratio',
    # Negation
    'neg_premise', 'neg_hypothesis', 'neg_mismatch',
    # WordNet
    'synonym_overlap', 'antonym_overlap', 'hypernym_overlap',
    'avg_wup_sim', 'max_wup_sim',
    # Sentiment
    'sentiment_diff', 'abs_sentiment_diff', 'polarity_mismatch',
]

# number of features in each group (for ablation)
# the numbers are off because I removed the POS features, but I'll keep the original numbering for consistency with the code and comments
FEATURE_GROUPS = {
    'lexical_overlap': [0, 1, 2, 3, 4, 5, 6],
    'string_alignment': [7, 8, 9, 10],
    'structural': [11, 12, 13, 14, 15, 16, 17],
    'negation': [18, 19, 20],
    'wordnet': [21, 22, 23, 24, 25],
    'sentiment': [30, 31, 32],
}

In [ ]:
print("Computing enhanced features for train set...")
hc_train = enhanced_handcrafted_features(train_df)
print(f"Train handcrafted shape: {hc_train.shape}")

print("\nComputing enhanced features for dev set...")
hc_dev = enhanced_handcrafted_features(dev_df)
print(f"Dev handcrafted shape: {hc_dev.shape}")

print(f"\nTotal handcrafted features: {len(FEATURE_NAMES)}")

### Build rich feature matrix (TFIDF + handcrafted)

In [ ]:
tfidf_p = TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, sublinear_tf=True)
tfidf_h = TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, sublinear_tf=True)

X_p_train = tfidf_p.fit_transform(train_df['premise'].fillna(''))
X_h_train = tfidf_h.fit_transform(train_df['hypothesis'].fillna(''))
X_p_dev   = tfidf_p.transform(dev_df['premise'].fillna(''))
X_h_dev   = tfidf_h.transform(dev_df['hypothesis'].fillna(''))

tfidf_shared = TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, sublinear_tf=True)
tfidf_shared.fit(pd.concat([train_df['premise'], train_df['hypothesis']]).fillna(''))

Xsp_train = tfidf_shared.transform(train_df['premise'].fillna(''))
Xsh_train = tfidf_shared.transform(train_df['hypothesis'].fillna(''))
Xsp_dev   = tfidf_shared.transform(dev_df['premise'].fillna(''))
Xsh_dev   = tfidf_shared.transform(dev_df['hypothesis'].fillna(''))

X_diff_train = Xsp_train - Xsh_train
X_prod_train = Xsp_train.multiply(Xsh_train)
X_diff_dev   = Xsp_dev - Xsh_dev
X_prod_dev   = Xsp_dev.multiply(Xsh_dev)

scaler = StandardScaler()
hc_train_scaled = scaler.fit_transform(hc_train)
hc_dev_scaled   = scaler.transform(hc_dev)

X_rich_train = sp.hstack([
    X_p_train, X_h_train,
    X_diff_train, X_prod_train,
    sp.csr_matrix(hc_train_scaled)
])
X_rich_dev = sp.hstack([
    X_p_dev, X_h_dev,
    X_diff_dev, X_prod_dev,
    sp.csr_matrix(hc_dev_scaled)
])

print(f"Rich feature matrix — train: {X_rich_train.shape}, dev: {X_rich_dev.shape}")

## 3. Model Comparison
We compare Logistic Regression, LinearSVC, and **Gradient Boosting**
(XGBoost-style via sklearn). Gradient boosting can learn non-linear
feature interactions that linear models miss — e.g. "high word overlap
AND negation mismatch" as a strong contradiction signal.

In [ ]:
def train_eval(name, model, X_tr, X_dv, y_tr, y_dv):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_dv)
    acc = accuracy_score(y_dv, y_pred)
    elapsed = time.time() - t0
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}  |  Time: {elapsed:.1f}s")
    print(classification_report(y_dv, y_pred))
    return model, y_pred, acc

results = {}

# Baseline (already trained)
results["LR baseline (concat TF-IDF)"] = accuracy_score(y_dev, y_pred_base)

# LR with rich features
lr_rich = LogisticRegression(max_iter=1000, C=1.0)
_, y_pred_lr, acc_lr = train_eval(
    "LR (rich features)", lr_rich,
    X_rich_train, X_rich_dev, y_train, y_dev
)
results["LR (rich features)"] = acc_lr

# LinearSVC with rich features
svc = LinearSVC(max_iter=3000, C=1.0)
_, y_pred_svc, acc_svc = train_eval(
    "LinearSVC (rich features)", svc,
    X_rich_train, X_rich_dev, y_train, y_dev
)
results["LinearSVC (rich features)"] = acc_svc

# Gradient Boosting — uses ONLY the handcrafted features (dense)
# because tree-based models don't benefit from high-dimensional sparse TF-IDF
# the same way linear models do.
print("\n--- Training GradientBoosting on handcrafted features only ---")
gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
_, y_pred_gb, acc_gb = train_eval(
    "GradientBoosting (handcrafted only)", gb,
    hc_train_scaled, hc_dev_scaled, y_train, y_dev
)
results["GradientBoosting (handcrafted)"] = acc_gb

## 4. Stacking Ensemble

In [ ]:
# For stacking, we combine predictions from:
# - LR on rich features (linear, handles sparse well)
# - GradientBoosting on handcrafted features (non-linear interactions)
#
# Simple approach: concatenate their predicted probabilities as meta-features

from sklearn.calibration import CalibratedClassifierCV

# Get probability estimates from each base model
# LR already has predict_proba; SVC needs calibration
lr_probs_train = cross_val_score(
    LogisticRegression(max_iter=1000, C=1.0),
    X_rich_train, y_train, cv=3, scoring='accuracy'
)
print(f"LR 3-fold CV accuracy: {lr_probs_train.mean():.4f} ± {lr_probs_train.std():.4f}")

# Simple stacking: train base models, get dev predictions, combine
# Using cross-validated predictions for train set to avoid data leakage
from sklearn.model_selection import StratifiedKFold

def get_oof_predictions(model, X, y, X_test, n_folds=5):
    """Get out-of-fold predictions for stacking (avoids data leakage)."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(y))
    test_preds = np.zeros(X_test.shape[0])

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        if sp.issparse(X):
            X_tr, X_val = X[tr_idx], X[val_idx]
        else:
            X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr = y.iloc[tr_idx] if hasattr(y, 'iloc') else y[tr_idx]

        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_tr, y_tr)

        if hasattr(model_clone, 'predict_proba'):
            oof_preds[val_idx] = model_clone.predict_proba(X_val)[:, 1]
            test_preds += model_clone.predict_proba(X_test)[:, 1] / n_folds
        else:
            oof_preds[val_idx] = model_clone.decision_function(X_val)
            test_preds += model_clone.decision_function(X_test) / n_folds

    return oof_preds, test_preds

print("Computing OOF predictions for stacking...")

# Base model 1: LR on rich features
oof_lr, test_lr = get_oof_predictions(
    LogisticRegression(max_iter=1000, C=1.0),
    X_rich_train, y_train, X_rich_dev
)

# Base model 2: GradientBoosting on handcrafted features
oof_gb, test_gb = get_oof_predictions(
    GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                subsample=0.8, random_state=42),
    hc_train_scaled, y_train, hc_dev_scaled
)

# Stack: combine OOF predictions as meta-features
meta_train = np.column_stack([oof_lr, oof_gb])
meta_test  = np.column_stack([test_lr, test_gb])

# Meta-learner
meta_lr = LogisticRegression(max_iter=1000)
meta_lr.fit(meta_train, y_train)
y_pred_stack = meta_lr.predict(meta_test)
acc_stack = accuracy_score(y_dev, y_pred_stack)

print(f"\n=== Stacking Ensemble ===")
print(f"Accuracy: {acc_stack:.4f}")
print(classification_report(y_dev, y_pred_stack))
results["Stacking Ensemble"] = acc_stack

## 5. Hyperparameter Tuning — Best Model

In [ ]:
# Tune LR on rich features with broader parameter grid
from sklearn.pipeline import Pipeline

param_dist = {
    'C': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
}

rs = RandomizedSearchCV(
    LogisticRegression(max_iter=2000),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42,
)
rs.fit(X_rich_train, y_train)

print(f"\nBest params: {rs.best_params_}")
print(f"Best CV accuracy: {rs.best_score_:.4f}")

best_model = rs.best_estimator_
y_pred_tuned = best_model.predict(X_rich_dev)
acc_tuned = accuracy_score(y_dev, y_pred_tuned)
print(f"Dev accuracy (tuned LR): {acc_tuned:.4f}")
print(classification_report(y_dev, y_pred_tuned))
results["LR (tuned)"] = acc_tuned

## 6. Ablation Study

In [ ]:
def build_feature_matrix_ablated(X_p, X_h, X_diff, X_prod, hc_scaled, exclude_group=None):
    """Build the rich feature matrix, optionally excluding a handcrafted feature group."""
    if exclude_group is not None:
        exclude_indices = FEATURE_GROUPS[exclude_group]
        keep_indices = [i for i in range(hc_scaled.shape[1]) if i not in exclude_indices]
        hc_ablated = hc_scaled[:, keep_indices]
    else:
        hc_ablated = hc_scaled

    return sp.hstack([
        X_p, X_h,
        X_diff, X_prod,
        sp.csr_matrix(hc_ablated)
    ])

print("=== Ablation Study: Removing one feature group at a time ===\n")

# Full model accuracy (using tuned LR params)
best_params = rs.best_params_
full_acc = acc_tuned
print(f"Full model accuracy: {full_acc:.4f}\n")

ablation_results = {}
for group_name in FEATURE_GROUPS:
    X_abl_train = build_feature_matrix_ablated(
        X_p_train, X_h_train, X_diff_train, X_prod_train,
        hc_train_scaled, exclude_group=group_name
    )
    X_abl_dev = build_feature_matrix_ablated(
        X_p_dev, X_h_dev, X_diff_dev, X_prod_dev,
        hc_dev_scaled, exclude_group=group_name
    )

    model_abl = LogisticRegression(**best_params, max_iter=2000)
    model_abl.fit(X_abl_train, y_train)
    abl_acc = accuracy_score(y_dev, model_abl.predict(X_abl_dev))
    drop = full_acc - abl_acc
    ablation_results[group_name] = {'accuracy': abl_acc, 'drop': drop}
    print(f"  Without {group_name:20s}: {abl_acc:.4f}  (drop: {drop:+.4f})")

# Also test without ALL handcrafted features (TF-IDF only)
X_tfidf_only_train = sp.hstack([X_p_train, X_h_train, X_diff_train, X_prod_train])
X_tfidf_only_dev   = sp.hstack([X_p_dev, X_h_dev, X_diff_dev, X_prod_dev])
model_tfidf = LogisticRegression(**best_params, max_iter=2000)
model_tfidf.fit(X_tfidf_only_train, y_train)
tfidf_only_acc = accuracy_score(y_dev, model_tfidf.predict(X_tfidf_only_dev))
print(f"\n  TF-IDF only (no handcrafted): {tfidf_only_acc:.4f}  (drop: {full_acc - tfidf_only_acc:+.4f})")

### Ablation Study bar chart

In [ ]:
groups = list(ablation_results.keys()) + ['ALL handcrafted']
drops = [ablation_results[g]['drop'] for g in ablation_results] + [full_acc - tfidf_only_acc]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d32f2f' if d > 0 else '#388e3c' for d in drops]
bars = ax.barh(groups, drops, color=colors, edgecolor='white')
ax.set_xlabel('Accuracy drop when group is removed')
ax.set_title('Ablation Study: Feature Group Contributions')
ax.axvline(0, color='black', linewidth=0.5)
for bar, drop in zip(bars, drops):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{drop:+.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig("ablation_study.png", dpi=150)
plt.show()


## 7. Statistical Significance Testing

In [ ]:
from scipy.stats import chi2

def mcnemar_test(y_true, y_pred_a, y_pred_b, model_a_name="Model A", model_b_name="Model B"):
    """
    McNemar's test for paired binary predictions.
    Tests whether two models make significantly different errors.
    """
    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)

    # Contingency: A right & B wrong, A wrong & B right
    b = np.sum(correct_a & ~correct_b)  # A right, B wrong
    c = np.sum(~correct_a & correct_b)  # A wrong, B right

    # McNemar's test with continuity correction
    if b + c == 0:
        print(f"  McNemar's test: No disagreements between {model_a_name} and {model_b_name}")
        return 1.0

    chi2_stat = (abs(b - c) - 1)**2 / (b + c)
    p_value = 1 - chi2.cdf(chi2_stat, df=1)

    print(f"  McNemar's test: {model_a_name} vs {model_b_name}")
    print(f"    {model_a_name} right, {model_b_name} wrong: {b}")
    print(f"    {model_a_name} wrong, {model_b_name} right: {c}")
    print(f"    χ² = {chi2_stat:.4f},  p = {p_value:.6f}")
    if p_value < 0.05:
        print(f"    → Significant at α = 0.05 ✓")
    else:
        print(f"    → NOT significant at α = 0.05")
    return p_value


def bootstrap_ci(y_true, y_pred, n_bootstrap=10000, ci=0.95, random_state=42):
    """
    Compute bootstrap confidence interval for accuracy.
    """
    rng = np.random.RandomState(random_state)
    n = len(y_true)
    scores = []
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        scores.append(accuracy_score(y_true_arr[idx], y_pred_arr[idx]))
    scores = np.array(scores)
    lower = np.percentile(scores, (1 - ci) / 2 * 100)
    upper = np.percentile(scores, (1 + ci) / 2 * 100)
    return np.mean(scores), lower, upper

In [ ]:
print("=== Statistical Significance Tests ===\n")

# Compare our best model vs baseline
y_pred_best = best_model.predict(X_rich_dev)

print("1. Our best LR (tuned, rich features) vs our LR baseline:")
mcnemar_test(y_dev, y_pred_best, y_pred_base,
             "LR_rich_tuned", "LR_baseline")

print()

# Compare our best vs GradientBoosting
print("2. Our best LR vs GradientBoosting (handcrafted):")
mcnemar_test(y_dev, y_pred_best, y_pred_gb,
             "LR_rich_tuned", "GradientBoosting")

print()

# Compare stacking vs best single model
print("3. Stacking Ensemble vs best single model (LR tuned):")
mcnemar_test(y_dev, y_pred_stack, y_pred_best,
             "Stacking", "LR_rich_tuned")

In [ ]:
print("\n=== Bootstrap 95% Confidence Intervals ===\n")

models_for_ci = {
    "LR baseline": y_pred_base,
    "LR (tuned, rich)": y_pred_best,
    "GradientBoosting": y_pred_gb,
    "Stacking Ensemble": y_pred_stack,
}

for name, preds in models_for_ci.items():
    mean_acc, lower, upper = bootstrap_ci(y_dev, preds)
    print(f"  {name:25s}: {mean_acc:.4f}  [{lower:.4f}, {upper:.4f}]")

## 8. Stratified Error Analysis
Analyse how accuracy varies across different data characteristics
to understand model failure modes.

In [ ]:
# Build analysis dataframe
analysis_df = dev_df.copy()
analysis_df['predicted'] = y_pred_best
analysis_df['correct'] = (analysis_df['predicted'] == analysis_df['label'])

# Add feature values for stratification
analysis_df['p_len'] = analysis_df['premise'].apply(lambda x: len(tokenize_simple(x)))
analysis_df['h_len'] = analysis_df['hypothesis'].apply(lambda x: len(tokenize_simple(x)))
analysis_df['word_overlap'] = analysis_df.apply(
    lambda r: len(set(tokenize_simple(r['premise'])) & set(tokenize_simple(r['hypothesis'])))
    / max(len(set(tokenize_simple(r['hypothesis']))), 1), axis=1
)
analysis_df['has_negation_mismatch'] = analysis_df.apply(
    lambda r: int(
        bool(set(tokenize_simple(r['premise'])) & NEGATION_WORDS) !=
        bool(set(tokenize_simple(r['hypothesis'])) & NEGATION_WORDS)
    ), axis=1
)

In [ ]:
print("=== Stratified Error Analysis ===\n")

# 1. By hypothesis length buckets
analysis_df['h_len_bucket'] = pd.cut(analysis_df['h_len'],
    bins=[0, 5, 10, 20, 50, 999],
    labels=['1-5', '6-10', '11-20', '21-50', '50+']
)

print("Accuracy by hypothesis length:")
for bucket in analysis_df['h_len_bucket'].cat.categories:
    subset = analysis_df[analysis_df['h_len_bucket'] == bucket]
    if len(subset) > 0:
        acc = subset['correct'].mean()
        print(f"  {bucket:10s}: {acc:.4f}  (n={len(subset)})")

# 2. By word overlap quartiles
analysis_df['overlap_quartile'] = pd.qcut(analysis_df['word_overlap'], 4,
    labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'], duplicates='drop')

print("\nAccuracy by word overlap quartile:")
for q in analysis_df['overlap_quartile'].cat.categories:
    subset = analysis_df[analysis_df['overlap_quartile'] == q]
    if len(subset) > 0:
        acc = subset['correct'].mean()
        print(f"  {q:15s}: {acc:.4f}  (n={len(subset)})")

# 3. By negation mismatch
print("\nAccuracy by negation mismatch:")
for val, label in [(0, 'No mismatch'), (1, 'Negation mismatch')]:
    subset = analysis_df[analysis_df['has_negation_mismatch'] == val]
    if len(subset) > 0:
        acc = subset['correct'].mean()
        print(f"  {label:20s}: {acc:.4f}  (n={len(subset)})")

# 4. By label
print("\nAccuracy by true label:")
for lab in [0, 1]:
    subset = analysis_df[analysis_df['label'] == lab]
    acc = subset['correct'].mean()
    print(f"  Label {lab}: {acc:.4f}  (n={len(subset)})")

In [ ]:
# Confusion matrix plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Best model
cm1 = confusion_matrix(y_dev, y_pred_best)
disp1 = ConfusionMatrixDisplay(cm1, display_labels=['Not Entailed', 'Entailed'])
disp1.plot(ax=axes[0], cmap='Blues')
axes[0].set_title(f'LR Tuned (Rich Features)\nAcc: {acc_tuned:.4f}')

# Baseline
cm2 = confusion_matrix(y_dev, y_pred_base)
disp2 = ConfusionMatrixDisplay(cm2, display_labels=['Not Entailed', 'Entailed'])
disp2.plot(ax=axes[1], cmap='Oranges')
axes[1].set_title(f'LR Baseline (concat TF-IDF)\nAcc: {accuracy_score(y_dev, y_pred_base):.4f}')

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

## 9. Summary

In [ ]:
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:40s}: {acc:.4f}")

print(f"\nBest model: LR (tuned) with {len(FEATURE_NAMES)} handcrafted features")
print(f"Improvement over concat TF-IDF baseline: "
      f"+{results.get('LR (tuned)', acc_tuned) - results['LR baseline (concat TF-IDF)']:.4f}")

## 10. Save Best Model Artefacts

In [ ]:
# the best model is the tuned LR with rich features. 
# for stacking, we could also save the base models and meta-learner, but for simplicity we'll just save the best single model and the vectorizers/scaler needed to reproduce its features.

import joblib

joblib.dump(best_model,    "NLI_model.joblib")
joblib.dump(tfidf_p,       "NLI_tfidf_premise.joblib")
joblib.dump(tfidf_h,       "NLI_tfidf_hypothesis.joblib")
joblib.dump(tfidf_shared,  "NLI_tfidf_shared.joblib")
joblib.dump(scaler,        "NLI_scaler.joblib")

print("Saved all model artefacts.")